# Main.py for testing


In [ ]:
from models.GAT.gat_journey_planner import create_pyg_graph
from pandasql import sqldf
import pandas as pd

def  load_pyg_data_t(start_des, final_des):
    pro_dir = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Main_App\\"
    data_path = "models\\GAT\\data\\"
    metro_edges_path = pro_dir+data_path+"pune_maas_journey_planner_data.csv"
    station_features_path = pro_dir+data_path+"station_features.csv"
    
    
    #loading in csv file
    route_edges_df = pd.read_csv(metro_edges_path) #pune_maas_journey_planner_data.csv - metro routing data table
    station_feat_df = pd.read_csv(station_features_path) #timetable_data.csv - metro train timetable data

    # clean columns data
    route_edges_df.columns = route_edges_df.columns.str.strip()
    station_feat_df.columns = station_feat_df.columns.str.strip()

    # fix differnt name - normalize station
    def clean_station_name(s):
        if pd.isna(s):
            return s
        return str(s).strip()

    # Clean edge df station names
    route_edges_df["station_name_from"] = route_edges_df["station_name_from"].apply(clean_station_name)
    route_edges_df["station_name_to"] = route_edges_df["station_name_to"].apply(clean_station_name)
    station_feat_df["station_name"] = station_feat_df["station_name"].apply(clean_station_name)
    
    # Map alternate names to one standard name
    station_name_map = {
        "RamWadi": "Ramwadi",
        "Ruby Hall": "Ruby Hall Clinic",
        "Civil Court": "District Court (Civil Court)",
        "District Court Pune": "District Court (Civil Court)",
    }

    def apply_station_map(s):
        if pd.isna(s):
            return s
        return station_name_map.get(s, s)


    #encode string attributes
    #for every new value map to unique 
    line_map = {name: i for i, name in enumerate(route_edges_df["line"].unique())}     #line: purple | pink | aqua
    mode_map = {name: i for i, name in enumerate(route_edges_df["mode"].unique(), start=1)} # mode: metro | feeder bus

    route_edges_df["line_id"] = route_edges_df["line"].map(line_map)
    route_edges_df["mode_id"] = route_edges_df["mode"].map(mode_map)
    route_edges_df["is_transfer"] = route_edges_df["is_transfer"].astype(int)
    route_edges_df["bidirectional"] = route_edges_df["bidirectional"].astype(int)
    
    #check
    # print ("========route_edges table========")
    # print (route_edges_df)

    # print ("\n========timetable table========")
    # print (timetable_df)

    # print ("\n========swipes metadata table========")
    # print (swipes_df)
    
    #convert df to pyg df
    pyg_data = create_pyg_graph(route_edges_df,station_feat_df)
    
    #get start/final destation id from df
    # query = """
    # SELECT station_id_to
    # FROM route_edges_df
    # WHERE station_name_to = 'Nigdi'
    # """
    # station_id_to = sqldf(query, {"route_edges_df": route_edges_df})['station_id_to'][0]
    
    # query = """
    # SELECT station_id_from
    # FROM route_edges_df
    # WHERE station_name_from = 'Chinchwad'
    # """
    # station_id_from = sqldf(query, {"route_edges_df": route_edges_df})['station_id_from'][0]
    
    # station_id_to = route_edges_df.loc[
    # route_edges_df["station_name_to"] == "Nigdi", "station_id_to"
    # ].iloc[0]

    # station_id_from = route_edges_df.loc[
    #     route_edges_df["station_name_from"] == "Chinchwad", "station_id_from"
    # ].iloc[0]

    
    # print("-------station_id_from--------\n")
    # print(station_id_from)
    
    # print("-------station_id_to--------\n")
    # print(station_id_to)
    
        
    return route_edges_df, station_feat_df, pyg_data#, station_id_to,station_id_from

load_pyg_data_t("start_des", "final_des")



In [ ]:
from pipeline.pipelines import build_pyg_graph,run_multitask_inference, add_readable_labels, build_adjacency_list,generate_routes,score_routes
# from models.GAT.gat_journey_planner import create_pyg_graph

def recommend_routes_t(origin_station,destination_station,metro_edges_df,station_features_df,model, max_transfer):
    # Build graph
    pyg_data = build_pyg_graph(metro_edges_df, station_features_df)

    #Run GAT predictions
    predictions_df, _, _ = run_multitask_inference(model, pyg_data)
    
    # print("===========================\n")
    # print("predictions_df:\n")
    # print(predictions_df)
    predictions_df = add_readable_labels(predictions_df)

    #Build route adjacency
    adjacency = build_adjacency_list(metro_edges_df)

    #Generate possible routes
    possible_routes = generate_routes(adjacency,origin_station,destination_station,max_transfer,max_routes=5)

    if not possible_routes:
        return {
            "origin_station": origin_station,
            "destination_station": destination_station,
            "routes": [],
            "message": "No candidate routes found."
        }

    #Score routes
    rated_routes = score_routes(possible_routes, predictions_df)

    return {
        "origin_station": origin_station,
        "destination_station": destination_station,
        "routes": rated_routes
    }


## current testing 

In [ ]:
import re
bot_response = "'{start_station: Bhakti Shakti, end_station: Ruby Hall Clinic,feeder_required: false,feeder_type: null,departure_time: null,arrival_time: null}'"

print(f"Bot: {bot_response}\n")

cleaned_res = re.split(r'([{}])', bot_response)

print(cleaned_res)

In [4]:
from pipeline.pipelines import recommend_routes, format_route_suggestions, normalize_trip_info, get_missing_fields,build_followup_question,congestion_penalty,feeder_bonus,get_Station_names
from models.GAT.gat_journey_planner import MultiTaskGAT
from datetime import datetime   
from models.llm.chatbot_agent import Chatbot, Extraction_SYS_MSG, FROMATTING_SYS_MSG
import re

TRIP_SCHEMA = {
        "start_station": None,
        "end_station": None,
        "start_station_id": None,
        "end_station_id": None,
        "feeder_required": None,   # true / false / null
        "feeder_type": None,       # bike / bus / shuttle / null
        "departure_time": None,
        "arrival_time": None
        # "start_location": None,
        # "final_destination": None
    }
pro_dir = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Main_App\\"
REQUIRED_FIELDS = ["start_station", "end_station", "feeder_required"]
checkpoint_path = pro_dir+"models\\checkpoint\\gat_maas_model.pt"
data_path = "models\\GAT\\data\\"
   

def test():
    metro_edges_path = pro_dir+data_path+"pune_maas_journey_planner_data.csv"
    station_features_path = pro_dir+data_path+"station_features.csv"
    recieved_required = False
    extraction_bot = Chatbot(Extraction_SYS_MSG,model="qwen2.5:7b")
    formatter_bot = Chatbot(FROMATTING_SYS_MSG,model="qwen2.5:7b")
    normalize_res = ""
                
    #user input
    request_mess = """
“Let’s plan your trip\n\n
First, I’ll need a few details:

What station are you starting from?
What station are you heading to?
Do you need a feeder service (bike, bus, etc.)?\n

You can also include:
departure or arrival time
your exact starting or final destination
    """
    print(request_mess)
    
    # user_input = "Need to get from Bhakti Shakti to Ruby Hall Clinic and yes i need a bike"#input("You: ")
    user_input = "Need to get from Bhakti Shakti to Ruby Hall Clinic and no i dont need a feeder"#input("You: ")
    
    # loop until user exist convo
    #while True:  
        
    # user_input = input("You: ")
                    
    #loop until all required info is recieved
    while True: 
        
        if user_input.lower() in ["exit", "quit"]:
            break
                        
        bot_response = extraction_bot.chat(user_input)
        # print(f"Bot: {bot_response}\n")
        
        print("RAW RESPONSE:", repr(bot_response))
        
        parse_res = re.split(r'([{}])', bot_response)
        cleaned_res = "{"+parse_res[2]+"}"
        print(cleaned_res)
                
        normalize_res = normalize_trip_info(cleaned_res)
        
        # if normalize_trip_info(cleaned_res) == None:
        missing_data, followup_required = get_missing_fields(normalize_res, REQUIRED_FIELDS)
            
        if not followup_required:
            print("All req info is obtained")
            break
        else:
            followup_questions = build_followup_question(missing_data)
            print("Bot:")
            print(followup_questions)
            user_input = input("You: ")
            
            # user_input = "follow up question:"+followup_questions + "and user response:"+ followup_rep
            # print(user_input)
    
    return normalize_res   
    # ------------


normalize_res = test()



“Let’s plan your trip


First, I’ll need a few details:

What station are you starting from?
What station are you heading to?
Do you need a feeder service (bike, bus, etc.)?


You can also include:
departure or arrival time
your exact starting or final destination
    
RAW RESPONSE: '{"start_station": "Bhakti Shakti", "end_station": "Ruby Hall Clinic","feeder_required": false,"feeder_type": null,"departure_time": null,"arrival_time": null}'
{"start_station": "Bhakti Shakti", "end_station": "Ruby Hall Clinic","feeder_required": false,"feeder_type": null,"departure_time": null,"arrival_time": null}
All req info is obtained


In [5]:
from pipeline.pipelines import load_pyg_data,load_trained_model,build_pyg_graph,run_multitask_inference,add_readable_labels,build_adjacency_list,generate_routes, load_data, write_data
 # test input
start_station_name = normalize_res["start_station"] #"Bhakti Shakti"#input("Start station>")
final_station_name = normalize_res["end_station"] #"Shivaji Nagar"#input("End station>")
# print("-------station_from--------\n")
# print(start_station_name)

# print("-------station_to--------\n")
# print(final_station_name)

# # metro_edges_df, station_features_df, pyg_gat_data ,station_id_to,station_id_from= load_pyg_data()
metro_edges_df, station_features_df, pyg_gat_data = load_pyg_data()

station_id_from = metro_edges_df.loc[
    metro_edges_df["station_name_from"] == start_station_name, "station_id_from"
    ].iloc[0]

station_id_to = metro_edges_df.loc[
    metro_edges_df["station_name_to"] == final_station_name, "station_id_to"
    ].iloc[0]


print("-------TRIP SCHEMA--------\n")
TRIP_SCHEMA["start_station"] = start_station_name
TRIP_SCHEMA["end_station"] = final_station_name
TRIP_SCHEMA["start_station_id"] = station_id_from
TRIP_SCHEMA["end_station_id"] = station_id_to
TRIP_SCHEMA["feeder_required"] = normalize_res["feeder_required"]
TRIP_SCHEMA["feeder_type"] = normalize_res["feeder_type"]
TRIP_SCHEMA["departure_time"] = normalize_res["departure_time"]
TRIP_SCHEMA["arrival_time"] = normalize_res["arrival_time"]      
print(TRIP_SCHEMA)

#update swipe db with user and trip details 
#then load model with updated info
#get best route
#then use that and time. if arrival time then determine what depart time the user will need to leave.
#if depart time then determine when arrive time
    #if feeder is required call service to be there by arrive time   
#user_info will be pulled from session data

def add_swipe_info(user_info,trip_info):   
         
    swipe_info = {
        "Name" : user_info["name"],
        "Age" : user_info["age"],
        "Gender" : user_info["gender"],
        "Start station" : trip_info["start_station"],
        "End Station" : trip_info["end_station"],
        "Start location - GPS" : None, #trip_info["gps_start_loc"],
        "End location GPS" : None, #trip_info["gps_final_loc"],
        "Approximate boarding time" : None,
        "Feeder Service (Y/N)" : "Y" if trip_info["feeder_required"] else "N",
        "Bike / Rickshaw" : None if trip_info["feeder_type"]==None else trip_info["feeder_type"].fillna(trip_info["feeder_type"]) ,
        "Swipe In station" : trip_info["start_station"],
        "Swipe in time":None
        } 
    print("-------------swipe info----------------")
    print(swipe_info)
    
    # update_swipe_db()
    #load swipe df to update file
    col = ["Name", "Age","Gender","Start station","End Station","Start location - GPS","End location GPS","Approximate boarding time","Feeder Service (Y/N)","Bike / Rickshaw","Swipe In station","Swipe in time"]
    swipe_csv_path = pro_dir+"data_bases\\swipe_metadata.csv"
    swipe_df = load_data(swipe_csv_path,col)
    
    swipe_df.loc[len(swipe_df)] = swipe_info
    # print("-------------swipe_df----------------")
    # print(swipe_df)
    #write it to the csv
    #write_data(swipe_csv_path,swipe_df)
        
user_info={
    "name":"test name",
    "age": "27",
    "gender":"female"}

add_swipe_info(user_info,TRIP_SCHEMA)

# result = test()


-------TRIP SCHEMA--------

{'start_station': 'Bhakti Shakti', 'end_station': 'Ruby Hall Clinic', 'start_station_id': 'P01', 'end_station_id': 'A14', 'feeder_required': False, 'feeder_type': None, 'departure_time': None, 'arrival_time': None}
-------------swipe info----------------
{'Name': 'test name', 'Age': '27', 'Gender': 'female', 'Start station': 'Bhakti Shakti', 'End Station': 'Ruby Hall Clinic', 'Start location - GPS': None, 'End location GPS': None, 'Approximate boarding time': None, 'Feeder Service (Y/N)': 'N', 'Bike / Rickshaw': None, 'Swipe In station': 'Bhakti Shakti', 'Swipe in time': None}
Read CSV successfully


In [30]:
import pandas as pd 
def get_time_table(station_line,station_name):
    db_path = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Main_App\\data_bases\\"
    tt_path = ""
    match station_line:
        case "purple":
            tt_path = db_path+"purple_line_timetable.csv"
            
        case "pink":
            tt_path = db_path+"pink_line_timetable.csv"
            
        case "aqua":
            tt_path = db_path+"aqua_line_timetable.csv"
            
    time_table_df = load_data(tt_path,None)
    
    return time_table_df
    
#get Approximate boarding time
def get_boarding_time(depart_time,station_line,station_name):
    found_in_list = False
    time  = pd.to_datetime(depart_time, format="%H:%M", errors="coerce")
    df = get_time_table(station_line,station_name)
    train_id = df["Train ID"].tolist()
    train_times = pd.to_datetime(df[station_name], format="%H:%M", errors="coerce").tolist()
    
    try:
        index = train_times.index(time)
        print(f"Found at index {index}")
        found_in_list = True
    except ValueError:
        print("Not found")
        found_in_list = False
        temp = train_times.copy()
        # add time to list
        temp.append(time)
        temp.sort()
        temp_index = temp.index(time)
        check = temp[temp_index+1]
        index = train_times.index(temp[temp_index+1])

    # print("-------------time-------------")
    # print(time)
    # print("-------------train_times------------t-")
    # print(train_times)
    # print("-------------dep index-------------")
    # print(index)
    # print("-------------found_in_list-------------")
    # print(found_in_list)
        
    if found_in_list:
        return {"train_id":train_id[index],"train_time":train_times[index]},{"train_id":train_id[index+1],"train_time":train_times[index+1]}
    else:
        return {"train_id":train_id[index],"train_time":train_times[index]},{"train_id":train_id[index+1],"train_time":train_times[index+1]}
    
def get_arrival_time(train_op,station_line,station_name,is_transfer):
    df = get_time_table(station_line,station_name)
    train_id = df["Train ID"].tolist()
    train_times = pd.to_datetime(df[station_name], format="%H:%M", errors="coerce").tolist()
    
    try:
        index = train_id.index(train_op["train_id"])
        print(f"Found at index {index}")
        
        time = train_times[index]
        
    except ValueError:
        print(f"Train id: {train_op["train_id"]} Not found")
        
    # print("-------------train_id-------------")
    # print(train_op["train_id"])
    # print("-------------train_times-------------")
    # print(train_times)
    # print("-------------index-------------")
    # print(index)
    # print("-------------time-------------")
    # print(time)
    return time
    
train_opt_a,train_opt_b =  get_boarding_time("6:11","aqua", "Chandni Chowk")
train_a_arr = get_arrival_time(train_opt_a,"aqua","Ideal Colony",False)
train_b_arr = get_arrival_time(train_opt_b,"aqua","Ideal Colony",False)

    

Read CSV successfully
Not found
Read CSV successfully
Found at index 2
Read CSV successfully
Found at index 3


In [ ]:
maas_gat_model, model_metadata = load_trained_model(MultiTaskGAT, checkpoint_path,model_kwargs ={
    "in_channels":pyg_gat_data.num_node_features,
    "hidden_channels":16, 
    "edge_features": pyg_gat_data.edge_attr.shape[1]
})

In [ ]:
# add for test
# def score_routes_t(possible_routes, predictions_df):
#     pred_map = predictions_df.set_index("station_id").to_dict("index")

#     scored_routes = []

#     for route in possible_routes:
#         total_congestion_penalty = 0.0
#         total_feeder_bonus = 0.0

#         station_details = []

#         for station_id in route:
#             station_pred = pred_map.get(station_id, None)

#             if station_pred is None:
#                 continue

#             c_class = station_pred["pred_congestion_class"]
#             f_class = station_pred["pred_feeder_class"]

#             total_congestion_penalty += congestion_penalty(c_class)
#             total_feeder_bonus += feeder_bonus(f_class)

#             station_details.append({
#                 "station_id": station_id,
#                 "congestion_label": station_pred["congestion_label"],
#                 "feeder_label": station_pred["feeder_label"],
#                 "congestion_confidence": station_pred["pred_congestion_confidence"],
#                 "feeder_confidence": station_pred["pred_feeder_confidence"],
#             })

#         route_length_penalty = len(route) - 1

#         final_score = route_length_penalty + total_congestion_penalty - total_feeder_bonus

#         # print("-------------route---------------")
#         # print(route)
#         station_names_route = get_Station_names(route)
#         # print("-------------station_names_route---------------")
#         # print(station_names_route)
        
#         scored_routes.append({
#             "route": station_names_route,
#             "num_stops": len(route),
#             "route_length_penalty": route_length_penalty,
#             "total_congestion_penalty": total_congestion_penalty,
#             "total_feeder_bonus": total_feeder_bonus,
#             "final_score": final_score,
#             "station_details": station_details
#         })

#     scored_routes.sort(key=lambda r: r["final_score"])
#     return scored_routes

# pyg_data = build_pyg_graph(metro_edges_df, station_features_df)
# # print("-------pyg_data--------\n")
# # print(pyg_data)
        
# #Run GAT predictions
# pred_df, _, _ = run_multitask_inference(maas_gat_model, pyg_data)
# pred_df = add_readable_labels(pred_df)
# # print("-------pred_df--------\n")
# # print(pred_df)

# #Build route adjacency
# adjacency = build_adjacency_list(metro_edges_df)
# # print("-------adjacency--------\n")
# # print(adjacency)

# #Generate possible routes
# possible_routes_ls = generate_routes(adjacency,TRIP_SCHEMA["start_station_id"],TRIP_SCHEMA["end_station_id"],max_routes=5)
# # print("-------possible_routes_ls--------\n")
# # print(possible_routes_ls)
# # print("end")

# ranked_routes = score_routes_t(possible_routes_ls, pred_df)

# print(ranked_routes)

In [ ]:
# from pipeline.pipelines import load_station_master
# def get_Station_names_t(route):
#     stations = []
    
#     station_master_df = load_station_master()

#     # for route_info in ranked_sugg_routes:
#     #     route_path = route_info #route_info["route"]
#     station_name = []
        
#         # for station_id in route_path:
#     for index in range(len(route)):      
#         station_id = route[index]
#         # print(station_id)
        
#         try: 
#             station_name.append(station_master_df.loc[
#                 station_master_df["master_station_id"] == station_id, "station_name"
#                 ].iloc[0])
            
#         except IndexError:
#             # No match found
#             # return "Station id: {station_id} was not found"
#             print(f"Station id: {station_id} was not found")
#         # if index == len(route)-1:#last index
#         #     # print(station_name)
#         #     stations.append(station_name)
            
#     # print(station_name)
#     return station_name

# test = ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'T002', 'T001', 'A12', 'A13', 'A14']
# st = get_Station_names_t(test)


In [ ]:
from pipeline.pipelines import recommend_routes
# end add
result = recommend_routes(
origin_station=TRIP_SCHEMA["start_station_id"],
destination_station=TRIP_SCHEMA["end_station_id"],
metro_edges_df=metro_edges_df,
station_features_df=station_features_df,
model=maas_gat_model
)

formatted_routes = format_route_suggestions(result)
print(formatted_routes)

In [ ]:
import pandas as pd
from pipeline.pipelines import load_station_master

sugg_routes_info_ls = result["routes"]
# pro_dir = "C:\\Users\\sasab\\Documents\\Projects\\MaaS_AI\\Main_App\\"
data_path = "models\\GAT\\data\\"
# print(sugg_routes_info_ls)
# fix differnt name - normalize station
# Map alternate names to one standard name
station_name_map = {
    "RamWadi": "Ramwadi",
    "Ruby Hall": "Ruby Hall Clinic",
    "Civil Court": "District Court (Civil Court)",
    "District Court Pune": "District Court (Civil Court)",
}

def get_Station_names_org(ranked_sugg_routes):
    stations = []
    
    station_master_df = load_station_master()

    for route_info in ranked_sugg_routes:
        route_path = route_info["route"]
        station_name = []
        
        # for station_id in route_path:
        for index in range(len(route_path)):      
            station_id = route_path[index]
            # print(station_id)
            
            try: 
                station_name.append(station_master_df.loc[
                    station_master_df["master_station_id"] == station_id, "station_name"
                    ].iloc[0])
                
            except IndexError:
                # No match found
                # return "Station id: {station_id} was not found"
                print(f"Station id: {station_id} was not found")
            
            if index == len(route_path)-1:#last index
                # print(station_name)
                stations.append(station_name)
                
    return stations    


st = get_Station_names(sugg_routes_info_ls) 
print(st)


# formatted_routes = format_route_suggestions(result)
# print(formatted_routes)